In [1]:
import os
import sys
import time
    
root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

from src.api.riot_client import RiotClient
from src.database.postgres import PostgresClient

In [2]:
API_KEY = os.getenv("RIOT_API_KEY")

client = RiotClient(api_key=API_KEY, game_name='coolguy', tag_line='super')

210540.4804286


In [3]:
game = client.get_ranked_match_ids(count_=1)[0]
print(game)

210543.4177309
NA1_5639245124


In [4]:
game_json = client.get_match_data(game)

# get metadata
metadata = {
    key: value 
    for key, value in game_json['info'].items() 
    if not isinstance(value, list) and not isinstance(value, dict)
}

# get game data
to_keep = [
    key for key in game_json['info']['participants'][0].keys() 
    if key not in ['PlayerBehavior', 'challenges', 'perks', 'missions']
]

game_data = {
    key: [] for key in to_keep
}
for i in range(10):
    curr_frame = game_json['info']['participants'][i]

    for key in to_keep:
        game_data[key].append(curr_frame[key])

game_data['championBanId'] = [x['championId']  for i in range(2) for x in game_json['info']['teams'][i]['bans']]

210546.2764976


In [6]:
db = PostgresClient()

with db.connect() as conn:
    with conn.cursor() as cursor:
        cursor.execute("SELECT current_database();")
        print(cursor.fetchone())

('league',)


In [61]:

from dotenv import load_dotenv
import os

load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

import psycopg
from psycopg.types.json import Jsonb

In [66]:
#POSTGRE TESTING

match_id = game 
payload = game_json

with psycopg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD
) as conn:
    # print('connected')

    # with conn.cursor() as cursor:
    #     cursor.execute("SELECT current_database();")
    #     print(cursor.fetchone())

    # print('closing connection')
    
    with conn.cursor() as cursor:
        cursor.execute(
            """
                SELECT
                    match_id,
                    payload->'metadata'->>'matchId' AS payload_match_id,
                    ingested_at
                FROM raw_matches;
            """
        )
        print(cursor.fetchone())

    # with conn.cursor() as cursor:
    #     cursor.execute(
    #         """
    #         INSERT INTO raw_matches (match_id, payload)
    #         VALUES (%s, %s) # %s how you pass values (match_id, Jsonb(game_json)) with psycopg
    #         """,
    #         (match_id, Jsonb(game_json)) # Jsonb turns game_json into a an actual json file not dictionary
    #     )

('NA1_5639245124', 'NA1_5639245124', datetime.datetime(2026, 9, 23, 20, 20, 10, 885043, tzinfo=zoneinfo.ZoneInfo(key='America/Los_Angeles')))
